In [1]:
import os
import sys
import glob
import logging
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

from typing import Literal
from pathlib import Path
from instanovo.utils.data_handler import SpectrumDataFrame

from instanovo.transformer.dataset import remove_modifications as clean_peptide

# Fix this later, imports should work without this
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))

[04/14/25 15:10:30] INFO     Enabling RDKit 2024.09.6 jupyter extensions                             ]8;id=558393;file:///home/hjisaac/.cache/pypoetry/virtualenvs/instanovoglyco-u5tn6RZG-py3.10/lib/python3.10/site-packages/rdkit/__init__.py\__init__.py]8;;\:]8;id=427199;file:///home/hjisaac/.cache/pypoetry/virtualenvs/instanovoglyco-u5tn6RZG-py3.10/lib/python3.10/site-packages/rdkit/__init__.py#22\22]8;;\

In [2]:
from common.utils import collect_files, get_or_create_folder, load_ipc_files
from common.logger import get_logger_config
from common.constants import (
    BASE_RAW_DATA_DIR,
    BASE_PROCESSED_DATA_DIR,
    BASE_LOGS_DIR,
    BASE_PLOTS_DIR,
    ROOT_DIR,
    BASE_REPORTS_CSV_DIR,
)

In [3]:
logger_config = get_logger_config(subdir="scripts")
logging.config.dictConfig(logger_config)
logger = logging.getLogger(__name__)

In [4]:
# Collect each unique_peptide.csv file
peptides_file_paths = [
    path
    for path in collect_files(BASE_REPORTS_CSV_DIR, ext="csv")
    if "unique_peptides" in path
]

assert peptides_file_paths, peptides_file_paths

In [14]:
df = pd.concat([pd.read_csv(file) for file in peptides_file_paths], ignore_index=True)
df.head(20)

,Unique Peptides
0,HNGTGGR
1,SQNCHNSSSR
2,AAGMNHTK
3,ANASHDQPQK
4,HNDSGASECR
5,GGGGGGGGGGGGGSGSSSGSSTSR
6,RQQQQQQQQQQQQK
7,QQQQQQQQQQQQK
8,KNDSGAYR
9,KCLNHTTQK


In [15]:
df["Unique Peptides"].describe()

count                163595
unique                44976
top       AVCMLSNTTAIAEAWAR
freq                     10
Name: Unique Peptides, dtype: object

## Split without Kevin constraint

In [16]:
unique_peptides_df = df["Unique Peptides"].drop_duplicates()

In [18]:
indices = np.arange(len(unique_peptides_df))
np.random.shuffle(indices)
split_ratio = 0.8

split_seperator = int(len(unique_peptides_df) * split_ratio)

# Shuffle the DataFrame indices
shuffled_df = unique_peptides_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Train/test split
train_peptides_df = shuffled_df.iloc[:split_seperator].reset_index(drop=True)
test_peptides_df = shuffled_df.iloc[split_seperator:].reset_index(drop=True)

In [19]:
assert len(train_peptides_df) == 35980, len(train_peptides_df)
assert len(test_peptides_df) == 8996, len(test_peptides_df)

In [20]:
def write_split(
    source_dir: Path | str,
    project_name: Path | str,
    split_name: Literal["train", "valid", "test"],  # noqa
    algorithm_version: Literal["v0", "v1", "v2", "v2.1"],
    potential_peptides_set: set,
    max_charge: int = 10,
    drop_unmodified: bool = False,
):
    file_paths = collect_files(location=source_dir)

    sdf, _ = load_ipc_files(file_paths)
    logger.info(f"Loaded {len(sdf)} entries from {source_dir}")

    # Filter by charge and peptide set
    sdf = sdf[
        (sdf["precursor_charge"] <= max_charge)
        & (sdf["precursor_charge"] > 0)
        & (sdf["peptide"].apply(lambda x: clean_peptide(x) in potential_peptides_set))
    ]
    logger.info(f"Got {len(sdf)} spectra after filtering by precursor charge")
    logger.info(f"Starting {split_name} split for project {project_name}")

    # Identify missing and fake modifications
    is_missing = sdf["modified_peptide"].isna()
    is_fake = sdf["modified_peptide"] == sdf["peptide"]

    logger.info(f"Found {is_missing.sum()} rows with missing modified_peptide")
    logger.info(
        f"Found {is_fake.sum()} rows with fake modified_peptide (same as peptide)"
    )

    # Treat fake modifications as unmodified
    is_unmodified = is_missing | is_fake

    if drop_unmodified:
        logger.info("Filtering out rows with missing or fake modified_peptide")
        sdf = sdf[~is_unmodified]
        logger.info(f"Left with {len(sdf)} rows after dropping unmodified rows")
    else:
        logger.info("Filling missing modified_peptide with related peptide")
        sdf.loc[is_missing, "modified_peptide"] = sdf["peptide"]

    assert (
        sdf["precursor_charge"].between(1, max_charge).all()
    ), "Some precursor_charge values are out of range."
    assert all(
        clean_peptide(p) in potential_peptides_set for p in sdf["peptide"]
    ), "Some peptides are not in the allowed set."
    assert (
        sdf["modified_peptide"].isna().sum() == 0
    ), "Every row should have modified_peptide set"

    # Save final file
    target_path = BASE_PROCESSED_DATA_DIR / project_name
    filename = f"dataset-ms-glyco_{algorithm_version}_{split_name}.parquet"
    sdf.to_parquet(path=target_path / filename, index=False)
    logger.info(
        f"Saved {len(sdf)} spectra for {split_name} to {target_path} for project {project_name}"
    )

In [4]:
projects_dirs = glob.glob(f"{BASE_RAW_DATA_DIR}/*/")
dirs_to_ignore = ["PXD044641_PXD035158"]  #

assert projects_dirs, projects_dirs

In [16]:
logger.info("Starting to split the dataset but using random split")

# Version 0 for train/test split: Constraint free split
# NOTE: This code is broken because of the param drop_unmodified
algorithm_version = "v0"
dirs_to_ignore = ["PXD044641_PXD035158"]  #
# DOCME: Replace the [] by projects_dirs to make the to script run
for project_dir in []:  # projects_dirs:
    project_name = project_dir.split("/")[-2]
    if project_name in dirs_to_ignore:
        logger.info(f"Skipping project {project_name} as part of projects to ignore")
        continue
    project_file_paths = collect_files(location=project_dir, ext="ipc")

    logger.info(
        f"Collected {len(project_file_paths)} of project {project_name} files from {project_dir}"
    )
    for split_name, peptide_set in [
        ("train", set(train_peptides_df)),
        # ("val", set(val_peptides_df)),
        ("test", set(test_peptides_df)),
    ]:
        write_split(
            project_name=project_name,
            split_name=split_name,
            algorithm_version=algorithm_version,
            potential_peptides_set=set(peptide_set),
            source_dir=f"{BASE_RAW_DATA_DIR / project_name}/",
        )

2025-04-14 11:56:58,254 - __main__ - INFO - Starting to split the dataset but using random split


In [23]:
kevin_train_peptides_array = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / "train_blacklist_overlap_identity_splits_massivekb_from_kevin_1067866_with_glyco_projects_44976_found_15499.csv"
)["Overlapped train peptides"].unique()
kevin_test_peptides_array = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / "test_overlap_identity_splits_massivekb_from_kevin_33575_with_glyco_projects_44976_found_4136.csv"
)["Overlapped test peptides"].unique()
kevin_val_peptides_array = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / "valid_overlap_identity_splits_massivekb_from_kevin_13062_with_glyco_projects_44976_found_495.csv"
)["Overlapped valid peptides"].unique()

In [ ]:
# Version 1 for train/test/valid split => peptide is used as fallback for modified_peptide
logger.info("Starting to split the dataset but taking into account kevin's suggestion")
dirs_to_ignore = ["PXD044641_PXD035158"]  #
# Focus on massivekb

algorithm_version = "v1"
# TODO: Uncomment the # projects_dirs to run the script
for project_dir in []:  # projects_dirs:
    project_name = project_dir.split("/")[-2]

    if project_name in dirs_to_ignore:
        logger.info(f"Skipping project {project_name} as part of projects to ignore")
        continue
    project_file_paths = collect_files(location=project_dir, ext="ipc")

    logger.info(
        f"Collected {len(project_file_paths)} of project {project_name} files from {project_dir}"
    )

    for split_name, kevin_peptide_set in [
        ("train", set(kevin_train_peptides_array)),
        ("valid", set(kevin_val_peptides_array)),
        ("test", set(kevin_test_peptides_array)),
    ]:
        write_split(
            drop_unmodified=False,
            project_name=project_name,
            split_name=split_name,
            algorithm_version=algorithm_version,
            potential_peptides_set=set(kevin_peptide_set),
            source_dir=f"{BASE_RAW_DATA_DIR / project_name}/",
        )

2025-04-10 00:41:01,638 - __main__ - INFO - Collected 27 of project PXD026629 files from /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/
2025-04-10 00:41:01,672 - __main__ - INFO - Instantiating SpectrumDataFrame with args=() and kwargs={'source': '/home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/*', 'source_type': 'ipc', 'column_mapping': {'intensity': 'intensity_array', 'mz': 'mz_array'}}
2025-04-10 00:41:01,676 - instanovo.utils.data_handler - INFO - Loading file 001 of 027: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/20180904YLJ-VSV4h-02.ipc
2025-04-10 00:41:01,882 - instanovo.utils.data_handler - INFO - Loading file 002 of 027: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/20180904YLJ-VSV0h-03.ipc
2025-04-10 00:41:02,004 - instanovo.utils.data_handler - INFO - Loading file 003 of 027: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/

In [5]:
# Version 2 or Version 2.1 for train/test/valid split => All rows with missing modified_peptides are filtered out. But is version 2.1 we also filter out fake modifications defined as modifications for which modified_peptide is equal to peptide.
logger.info("Starting to split the dataset but taking into account kevin's suggestion")
dirs_to_ignore = ["PXD044641_PXD035158"]  #

# Focus on massivekb
algorithm_version = "v2.1"  # Version 2.1
for project_dir in projects_dirs:  # projects_dirs:
    project_name = project_dir.split("/")[-2]

    if project_name in dirs_to_ignore:
        logger.info(f"Skipping project {project_name} as part of projects to ignore")
        continue
    project_file_paths = collect_files(location=project_dir, ext="ipc")

    logger.info(
        f"Collected {len(project_file_paths)} of project {project_name} files from {project_dir}"
    )

    for split_name, kevin_peptide_set in [
        ("train", set(kevin_train_peptides_array)),
        ("valid", set(kevin_val_peptides_array)),
        ("test", set(kevin_test_peptides_array)),
    ]:
        write_split(
            # The difference here
            drop_unmodified=True,
            project_name=project_name,
            split_name=split_name,
            algorithm_version=algorithm_version,
            potential_peptides_set=set(kevin_peptide_set),
            source_dir=f"{BASE_RAW_DATA_DIR / project_name}/",
        )

,index,scan,header,rt,frag_type,collision_energy,precursor_mz,precursor_charge,precursor_intensity,lower_offset,...,peptide_observed_mz,peptide_calc_mz,delta_mass,expectation,hyperscore,nextscore,probability,auc_intensity,protein,experiment_name
0,8949,controllerType=0 controllerNumber=1 scan=8950,FTMS + c NSI d Full ms2 839.3411@hcd33.00 [120...,1294.307551,HCD,33.0,838.839966,2,9.527561e+05,1.0,...,838.8400,838.8351,0.0029,0.149623,8.800,0.000,0.7580,3325199.8,sp|Q8BPN8|DMXL2_MOUSE,Fut8_WT_max_IGP_mousebrain_2.mzML
1,9337,controllerType=0 controllerNumber=1 scan=9338,FTMS + c NSI d Full ms2 1026.7343@hcd33.00 [12...,1350.437945,HCD,33.0,1026.399780,3,1.232893e+06,1.0,...,1026.3997,1026.3933,-0.0021,0.000077,22.207,15.165,0.9932,46660924.0,sp|P97300|NPTN_MOUSE,Fut8_WT_max_IGP_mousebrain_2.mzML
2,9500,controllerType=0 controllerNumber=1 scan=9501,FTMS + c NSI d Full ms2 972.7153@hcd33.00 [120...,1374.236220,HCD,33.0,972.381165,3,3.984886e+06,1.0,...,972.3811,972.3757,-0.0024,0.000124,24.663,13.121,0.9930,78170784.0,sp|P97300|NPTN_MOUSE,Fut8_WT_max_IGP_mousebrain_2.mzML
3,9505,controllerType=0 controllerNumber=1 scan=9506,FTMS + c NSI d Full ms2 1026.7322@hcd33.00 [12...,1375.107693,HCD,33.0,1026.399048,3,1.984562e+06,1.0,...,1026.3990,1026.3933,-0.0025,0.001988,14.423,7.244,0.9566,46660924.0,sp|P97300|NPTN_MOUSE,Fut8_WT_max_IGP_mousebrain_2.mzML
4,9545,controllerType=0 controllerNumber=1 scan=9546,FTMS + c NSI d Full ms2 1108.9292@hcd33.00 [12...,1380.829213,HCD,33.0,1108.929199,2,7.928026e+06,1.0,...,1108.9292,1108.9227,-0.0028,0.000097,25.557,11.495,0.9929,34914600.0,sp|P97300|NPTN_MOUSE,Fut8_WT_max_IGP_mousebrain_2.mzML


In [9]:
# rr["modified_peptide"].head(100)

0                    None
1     N[2191]ASNM[147]EYR
2     N[2029]ASNM[147]EYR
3     N[2191]ASNM[147]EYR
4     N[1330]ASNM[147]EYR
             ...         
95      FGTVPN[1817]GSTER
96      SIAHN[1493]MTTPNK
97          NLN[1330]FSTR
98         KN[1493]STAYFR
99         KN[1493]STAYFR
Name: modified_peptide, Length: 100, dtype: object

## Attempt to analyze the content of the split files

### Split Version 2.1

In [29]:
split_version = 2.1
logger.info(f"Split version {split_version} content analysis")

for split in ("train", "valid", "test"):
    dfs = []
    for project_dir in projects_dirs:  # projects_dirs:
        project_name = project_dir.split("/")[-2]

        if project_name in dirs_to_ignore:
            logger.info(
                f"Skipping project {project_name} as part of projects to ignore"
            )
            continue

        file_path = f"{project_name}/dataset-ms-glyco_v{split_version}_{split}.parquet"
        df = pd.read_parquet(BASE_PROCESSED_DATA_DIR / file_path)
        logger.info(f"Got {len(df)} rows from {file_path} for project {project_name}")
        dfs.append(df)

    result = pd.concat(dfs, ignore_index=True)
    logger.info(
        f"Overall {len(result)} rows for project {project_name} {split} v{split_version}"
    )

    result[["peptide", "modified_peptide"]].to_csv(
        BASE_REPORTS_CSV_DIR
        / f"version{split_version}_{split}_split_peptide_and_modified_peptides.csv",
        index=False,
    )

2025-04-14 18:02:08,645 - __main__ - INFO - Split version 2.1 content analysis
2025-04-14 18:02:14,786 - __main__ - INFO - Got 41292 rows from PXD026629/dataset-ms-glyco_v2.1_train.parquet for project PXD026629
2025-04-14 18:02:15,968 - __main__ - INFO - Got 24670 rows from PXD031032/dataset-ms-glyco_v2.1_train.parquet for project PXD031032
2025-04-14 18:02:19,106 - __main__ - INFO - Got 37352 rows from PXD031025/dataset-ms-glyco_v2.1_train.parquet for project PXD031025
2025-04-14 18:02:19,107 - __main__ - INFO - Skipping project PXD044641_PXD035158 as part of projects to ignore
2025-04-14 18:02:37,418 - __main__ - INFO - Got 251216 rows from PXD026649/dataset-ms-glyco_v2.1_train.parquet for project PXD026649
2025-04-14 18:02:39,517 - __main__ - INFO - Got 135622 rows from PXD047898/dataset-ms-glyco_v2.1_train.parquet for project PXD047898
2025-04-14 18:02:40,098 - __main__ - INFO - Got 11707 rows from PXD044641/dataset-ms-glyco_v2.1_train.parquet for project PXD044641
2025-04-14 18:02

In [30]:
train_peptide_and_modified_df = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / f"version{split_version}_train_split_peptide_and_modified_peptides.csv",
)

train_peptide_and_modified_df.describe()

,peptide,modified_peptide
count,619431,619431
unique,4335,24527
top,MVSHHNLTTGATLINEQWLLTTAK,YLGN[2319]ATAIFFLPDEGK
freq,22960,8675


In [31]:
valid_peptide_and_modified_df = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / f"version{split_version}_valid_split_peptide_and_modified_peptides.csv",
)
valid_peptide_and_modified_df.describe()

,peptide,modified_peptide
count,8715,8715
unique,117,490
top,TAVNCSSDFDACLITK,N[1655]FTEIASK
freq,2108,203


In [32]:
test_peptide_and_modified_df = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / f"version{split_version}_test_split_peptide_and_modified_peptides.csv",
)
test_peptide_and_modified_df.describe()

,peptide,modified_peptide
count,38607,38607
unique,705,1405
top,VVLHPNYSQVDIGLIK,VVLHPN[2319]YSQVDIGLIK
freq,9074,4514


In [33]:
split_version = 1
logger.info(f"Split version {split_version} content analysis")

for split in ("train", "valid", "test"):
    dfs = []
    for project_dir in projects_dirs:  # projects_dirs:
        project_name = project_dir.split("/")[-2]

        if project_name in dirs_to_ignore:
            logger.info(
                f"Skipping project {project_name} as part of projects to ignore"
            )
            continue
        logger.info(f"Reading {project_dir}")
        df = pd.read_parquet(
            "/home/hjisaac/AI4Science/instanovo_instadeep/glycodata_processed_version1/processed"
            f"/{project_name}/dataset-ms-glyco_v{split_version}_{split}.parquet"
        )
        dfs.append(df)

    result = pd.concat(dfs, ignore_index=True)

    result[["peptide", "modified_peptide"]].to_csv(
        BASE_REPORTS_CSV_DIR
        / f"version{split_version}_{split}_split_peptide_and_modified_peptides.csv",
        index=False,
    )

2025-04-14 18:02:56,826 - __main__ - INFO - Split version 1 content analysis
2025-04-14 18:02:56,831 - __main__ - INFO - Reading /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/
2025-04-14 18:02:58,647 - __main__ - INFO - Reading /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD031032/
2025-04-14 18:03:00,119 - __main__ - INFO - Reading /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD031025/
2025-04-14 18:03:01,993 - __main__ - INFO - Skipping project PXD044641_PXD035158 as part of projects to ignore
2025-04-14 18:03:01,993 - __main__ - INFO - Reading /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026649/
2025-04-14 18:03:10,717 - __main__ - INFO - Reading /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD047898/
2025-04-14 18:03:13,417 - __main__ - INFO - Reading /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD044641/
2025-04-14 18:03:13,

In [34]:
train_peptide_and_modified_df = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / f"version{split_version}_train_split_peptide_and_modified_peptides.csv",
)

train_peptide_and_modified_df.describe()

,peptide,modified_peptide
count,1066365,1066365
unique,15507,37333
top,EFNAETFTFHADICTLSEK,EFNAETFTFHADICTLSEK
freq,35163,35163


In [35]:
valid_peptide_and_modified_df = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / f"version{split_version}_valid_split_peptide_and_modified_peptides.csv",
)
valid_peptide_and_modified_df.describe()

,peptide,modified_peptide
count,16049,16049
unique,496,910
top,TAVNCSSDFDACLITK,VTEQLIEAISNGDFESYTK
freq,2108,621


In [36]:
test_peptide_and_modified_df = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / f"version{split_version}_test_split_peptide_and_modified_peptides.csv",
)
test_peptide_and_modified_df.describe()

,peptide,modified_peptide
count,324276,324276
unique,4141,5334
top,DVFLGMFLYEYAR,DVFLGMFLYEYAR
freq,24662,23379


## Investigation for Kostas

In [2]:
train1 = pd.read_csv(
    "/home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/.trash_local/version1_train_split_peptide_and_modified_peptides.csv"
)

In [4]:
len(train1["modified_peptide"].unique())

4885

In [7]:
len(train1)

151334

In [3]:
train2 = pd.read_csv(
    "/home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/.trash_local/version2.1_train_split_peptide_and_modified_peptides.csv"
)

In [6]:
len(train2)

50098

In [5]:
len(train2["modified_peptide"].unique())

2279